In [ ]:
import pandas as pd
from src.QC import *
from src.plotting_functions import *

hme1_1 = pd.read_csv("../../Experiment/1_hTERT_HME1/Data/Processed/20260505_hTERT_HME1_1_processes_phosphoPlus.tsv", sep="\t").fillna(0)
hme1_2 = pd.read_csv("../../Experiment/2_hTERT_HME1/Data/Processed/20260505_hTERT_HEM1_2_processes_phosphoPlus.tsv", sep="\t").fillna(0)
hek_1 = pd.read_csv("../../Experiment/1_HEK293T/Data/Processed/20260505_HEK293T_raw_selection_processes_phosphoPlus.tsv", sep="\t").fillna(0)
hme1_mutants = pd.read_csv("../../Experiment/3_hTERT_HME1_mutants_comparison/Data/Processed/20260505_hTERT_HME1_mutants_test_processed_phosphoPlus_filtered.tsv", sep="\t").fillna(0)

In [ ]:
TMT_plex_sample = hme1_2[['site',
                 'n:reps',
                 'WT_raw:abs_EGF_starve_r1',
                 'WT_raw:abs_EGF_starve_r2',
                 'WT_raw:abs_EGF_starve_r3',
                 'WT_raw:abs_EGF_starve_r4']]


# MSMS data quality check

In [ ]:
missing_values_per_sample(
    TMT_plex_sample,
    cell_lines=["WT"], # "BRAFS151A", "GAB1Y259A"],
    conditions=["_EGF_"], # "_INS_","_EGFnINS_"],
    data_type="raw:abs",
    missing_threshold=0.30,
    figsize=None,
    title="Missing values per sample",
    ax=None,
)

In [ ]:
peptide_count_per_sample(
    TMT_plex_sample,
    cell_lines=["WT"], #"BRAFS151A", "GAB1Y259A"],
    conditions=["_EGF_"], # "_INS_","_EGFnINS_"],
    data_type="raw:abs",
    missing_value=0.0,
    figsize=None,
    title="Detected peptides per sample",
    ax=None,
)

In [ ]:
nreps_distribution(
    TMT_plex_sample,
    nreps_col="n:reps",
    figsize=(6, 4),
    title="Replicate count distribution",
    ax=None,
)

In [ ]:
replicate_detection_map(
    TMT_plex_sample,
    cell_lines=["WT"],
    conditions=["_EGF_"], #, "_INS_","_EGFnINS_"],
    data_type="raw:abs",
    missing_value=0.0,
    figsize=None,
    title="Replicate detection map",
    ax=None,
)

In [ ]:
intensity_distribution(
    hme1_2,
    cell_lines=["WT"],
    conditions=["_EGF_"], #, "_INS_","_EGFnINS_"],
    data_type="log2:abs",
    log_transform= False,
    figsize=None,
    title="Intensity distribution per sample",
    ax=None,
)

In [ ]:
cv_distribution(
    hme1_2,
    cell_lines=["WT"],
    conditions=["_EGF_"], # "_INS_","_EGFnINS_"],
    data_type="raw:cv",
    cv_threshold=30.0,
    figsize=None,
    title="CV distribution per timepoint",
    ax=None,
)

In [ ]:
fig_scatter, fig_variance, pca_df = pca_plot_interactive(
                                                        hme1_2,
                                                        cell_lines=["WT"], # "BRAFS151A", "GAB1Y259A"],
                                                        conditions=["_EGF_"], #"_INS_","_EGFnINS_"],
                                                        data_type="log2:abs",
                                                        n_components=10,
                                                        impute=True,
                                                        impute_method="mean",
                                                        color_by="condition", #condition  cell_line
                                                        figsize=(1000, 800),
                                                        title="PCA of replicate samples",
                                                    )
fig_scatter.show()

In [ ]:
fig, umap = umap_plot_interactive(hme1_2,
                                  cell_lines=["WT"], # "BRAFS151A", "GAB1Y259A"],
                                  conditions=["_EGF_"], # "_INS_","_EGFnINS_"],
                                  data_type="raw:abs",
                                  impute=True,
                                  impute_method="mean",
                                  n_neighbors=15,
                                  min_dist=0.1,
                                  random_state=42,
                                  color_by="condition",
                                  figsize=(1000, 800),
                                  title="UMAP of replicate samples",)

fig.show()

In [ ]:
fig, dist_df, pca_df = pca_distance_heatmap(
                                            hme1_2,
                                            cell_lines=["WT"], #"WT", "BRAFS151A", "GAB1Y259A"],
                                            conditions=["_EGF_"], #"_INS_","_EGFnINS_"],
                                            data_type="log2:abs",
                                            impute=True,
                                            impute_method="mean",
                                            n_pca_components=3,
                                            centroid_stat="median",
                                            zmax=None,
                                            figsize=(600, 600),
                                            title="Distance between conditions (PCA centroid)",
                                        )
fig.show()

In [ ]:
fig, ax = plot_sites_umap_interactive(hme1_2,
                                    cell_lines=["WT"], #"WT", "BRAFS151A", "GAB1Y259A"],
                                    conditions=["_EGF_", "_INS_","_EGFnINS_"],
                                    data_type="log2:scaled",
                                    exclude_full=True,
                                    color_col=None,
                                    hover_cols=None,
                                    n_neighbors=15,
                                    min_dist=0.1,
                                    random_state=42,
                                    figsize=(1000, 800),
                                    title=None,
)


fig.show()

# Datasets comparison

Compare phosphosite identity across the four raw MSMS datasets:
- `hek_1` — HEK293T experiment 1
- `hme1_1` — hTERT-HME1 experiment 1
- `hme1_2` — hTERT-HME1 experiment 2
- `hme1_mutants` — hTERT-HME1 mutant cell lines

Analyses:
- Venn diagrams of shared phosphosites
- Pairwise overlap summary table (n detected, n shared, % overlap)
- WT vs mutant overlap (critical for classifier transfer)

## Venn diagrams

In [ ]:
# hTERT-HME1 replicate experiments + HEK293T
venn_diagrams(
    data_frames=[hek_1, hme1_1, hme1_2],
    labels=["HEK293T", "hTERT-HME1 exp1", "hTERT-HME1 exp2"],
    colors=["green", "blue", "red"],
    title="Phosphosite overlap — WT experiments",
    column_to_compare="site",
)

In [ ]:
# hTERT-HME1 WT (exp1) vs mutant dataset
# Critical for classifier transfer: only shared sites can be used to propagate cluster labels
venn_diagrams(
    data_frames=[hme1_1, hme1_mutants],
    labels=["hTERT-HME1 WT", "hTERT-HME1 mutants"],
    colors=["blue", "orange"],
    title="Phosphosite overlap — WT vs mutant (hTERT-HME1)",
    column_to_compare="site",
)

## Pairwise overlap summary

Quantitative table of n detected, n shared, and % overlap for every dataset pair.
`pct_A_in_B` = % of dataset A's sites also found in B (coverage from A's perspective).

In [ ]:
overlap_summary(
    data_frames=[hek_1, hme1_1, hme1_2],
    labels=["HEK293T", "hTERT-HME1 exp1", "hTERT-HME1 exp2"],
    column_to_compare="site",
)